In [ ]:
# Modules Installation
!pip install -q groq

In [ ]:
# Modules Import
import os
import sys
import datetime
import random
import time
import json

from groq import Groq

In [ ]:
cot_prompt = """You are Linguistic expert, which can generate chain of thoughts to identify relationship between entities in given the sentence.
Sentence: {}
Generate a paragraph with Chain of Thoughts in 2-3 short and simple sentences, which explain relationship between entities in the sentence.
Generate the response in the form: Relationship between entities {} and {} is {} because <chain of thought explanation>"""

In [ ]:
# Configurations

# SemEval2010-Task8 Dataset
#dataset = "semeval2010-task8-dataset"
#dataset_dir = "/kaggle/input/semeval2010-task8-dataset"

# TACRED Dataset
dataset = "tacred-dataset"
dataset_dir = "/kaggle/input/tacred-dataset"

model_name = "llama-3.3-70b-versatile"
             
prompt_ver = "cot_prompt"
prompt = eval(prompt_ver)

In [ ]:
# Save configuration in config.json
output_path = "/kaggle/working"

if model_name.count("/") > 1:
    path_model_name = model_name.split("/")[3].replace(".", "").strip()
elif  model_name.count("/") == 1:
    path_model_name = model_name.split("/")[1].replace(".", "").strip()
else:
    path_model_name = model_name
print(path_model_name)

output_path = os.path.join(output_path, dataset + "_" + path_model_name)

# Create folder with dataset_model name
if not os.path.exists(output_path):
    os.makedirs(output_path) 

In [ ]:
def get_examples(file):
    # open the JSON file
    with open(file) as f: 
        # read all json objects as example dictionaries and store in the example list
        examples_list = json.load(f)

        # Close the file
        f.close()

    # return list of examples: each example is dictionary
    return examples_list

In [ ]:
# Dataset initialization
dataset_trainfile = os.path.join(dataset_dir, "train.json")

list_train_examples = get_examples(dataset_trainfile)

In [ ]:
client = Groq(
    api_key="API_KEY",
    
)

In [ ]:
# Open output file to store results
file_name = os.path.join(output_path, 
            "result_" + model_name + "_" + dataset + "1.json")
fp1 = open(file_name, 'w')

# Run model and extract results
print("Processing Examples: ")
for index, example in enumerate(list_train_examples[35000:]):
    
    prompt_text = prompt.format(example['sentence'], example['e1'], example['e2'], example['relation'])
    
    try:
        chat_completion = client.chat.completions.create(
                                messages=[
                                    {
                                        "role": "user",
                                        "content": prompt_text,
                                    }
                                ],
                                model=model_name,
                                temperature=0.6,
                                max_tokens=200,
                                top_p=0.9,
                                stream=False)

    except:
        print("Exception Occured:")

    example['cot_explanation'] = chat_completion.choices[0].message.content
    
    print("Example: {id:6s} {e1:20s} {e2:20s} {rel:25s}".format(id=str(example['id']), e1=example['e1'], 
                    e2=example['e2'], rel=example['relation']))
    
    json.dump(example, fp1)    

fp1.close()

In [ ]:
# Save output dictionary in a file.
file_name = os.path.join(output_path, 
            "result_" + model_name + "_" + dataset + ".json")

with open(file_name, 'w') as fp:
    json.dump(list_train_examples, fp)
    fp.close()

In [ ]:
print("THE-END")